# Results

Load per-run JSON from `../results/`, build the comparison table and plots used in the README.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

RESULTS = Path('../results')
runs = [json.loads(p.read_text()) for p in sorted(RESULTS.glob('*.json'))]
print(f'{len(runs)} runs loaded')


## Headline accuracy / macro-F1 table

In [ ]:
rows = []
for r in runs:
    for split, metrics in r['splits'].items():
        rows.append({
            'task': r['task'], 'model': r['model'], 'split': split,
            'accuracy': metrics['accuracy'], 'macro_f1': metrics['macro_f1'],
        })
df = pd.DataFrame(rows)

table = df.pivot_table(index=['task', 'split'], columns='model',
                       values='accuracy').round(3) * 100
table


## Accuracy bar chart

In [ ]:
fig, axes = plt.subplots(1, df['task'].nunique(), figsize=(6 * df['task'].nunique(), 4), sharey=True)
if df['task'].nunique() == 1:
    axes = [axes]
for ax, task in zip(axes, sorted(df['task'].unique())):
    sub = df[df['task'] == task]
    sns.barplot(sub, x='model', y='accuracy', hue='split', ax=ax)
    ax.set_title(task)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Test accuracy')
    ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(RESULTS / 'summary_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()


## Per-class F1 (test split)

In [ ]:
rows = []
for r in runs:
    for cls, m in r['splits']['test']['per_class'].items():
        rows.append({'task': r['task'], 'model': r['model'], 'class': cls, 'f1': m['f1']})
pc = pd.DataFrame(rows)
pc.pivot_table(index=['task', 'class'], columns='model', values='f1').round(3)


## Training curves

In [ ]:
fig, axes = plt.subplots(len(runs), 2, figsize=(11, 2.6 * len(runs)))
if len(runs) == 1:
    axes = axes[None, :]
for i, r in enumerate(runs):
    h = pd.DataFrame(r['history'])
    h['x'] = range(1, len(h) + 1)
    axes[i, 0].plot(h['x'], h['train_acc'], label='train')
    axes[i, 0].plot(h['x'], h['val_acc'], label='val')
    axes[i, 0].set_title(f"{r['task']} / {r['model']} — accuracy")
    axes[i, 0].legend()
    axes[i, 1].plot(h['x'], h['train_loss'], label='train')
    axes[i, 1].plot(h['x'], h['val_loss'], label='val')
    axes[i, 1].set_title(f"{r['task']} / {r['model']} — loss")
    axes[i, 1].legend()
fig.tight_layout()
plt.show()
